# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsahmed39/FlyRank--Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is **classification, whose output I use as a ranking score**  not a straight up/down clustering or pure signal analysis question. The underlying task is will this content item's clicks keep falling in the near term , but the editor doesn't want a flat yes/no list they want the items sorted, because they can only review a fixed number this week. So the model outputs a probability, and I rank items by that probability. This matches the framing guide's table: "will this one decline?" maps to classification, but because the real decision is "which ones first," the probability gets consumed as a score, not just a label.

In [4]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")
eligible = df.dropna(subset=["clicks_prev_30d", "clicks_last_30d"])
eligible = eligible[eligible["clicks_prev_30d"] > 0]
print(f"Items an editor could plausibly review: {len(eligible):,}")
print("That's too many to sort by hand every week -- hence ranking, not a flat list.")

Items an editor could plausibly review: 11,759
That's too many to sort by hand every week -- hence ranking, not a flat list.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `recent_drop` a binary label I define myself: 1 if clicks fell 20% or more from `clicks_prev_30d` to `clicks_last_30d`, 0 otherwise, restricted to items that had clicks in `clicks_prev_30d`. So a drop is measurable.

**Where it comes from:** two real observed 30 day windows in the data, not FlyRank's `trend_direction` bucket That said, I should be careful here: this is still a proxy, not a true held out future outcome `last_30d` is the most recent window in a single static snapshot, not a window that comes strictly after some model training cutoff. A stronger version of this label, once I move to the warehouse's daily fact table, would use a genuine time split: features from one window, label from a later window that the model never saw at feature-build time.

In [5]:
sub = df.dropna(subset=["clicks_prev_30d", "clicks_last_30d"]).copy()
sub = sub[sub["clicks_prev_30d"] > 0]
sub["pct_change_30d"] = (sub["clicks_last_30d"] - sub["clicks_prev_30d"]) / sub["clicks_prev_30d"]
sub["recent_drop"] = (sub["pct_change_30d"] <= -0.20).astype(int)

print(sub["recent_drop"].value_counts())
print(f"\nBase rate (share currently labeled recent_drop=1): {sub['recent_drop'].mean():.1%}")

recent_drop
1    6147
0    5612
Name: count, dtype: int64

Base rate (share currently labeled recent_drop=1): 52.3%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.** An editor can realistically review about 50 items a week. So the number that matters isn't overall accuracy it's "of the top 50 items my ranking puts first, how many are actually still declining?" A high base rate means overall accuracy would be a weak, easily gamed metric here precision@K matches the actual decision capacity instead.

To show the metric isn't circular, I compute it below for a trivial, non model rule sort by raw prior traffic just to see what a naive ordering gets, before any real model exists. This is a mechanics check, not a claim about a working model.

In [6]:
naive_top50 = sub.sort_values("clicks_prev_30d", ascending=False).head(50)
naive_precision_at_50 = naive_top50["recent_drop"].mean()
print(f"Naive rule (sort by prior traffic only) precision@50: {naive_precision_at_50:.2f}")
print("A real model needs to beat this trivial ordering, not just beat 0.")

Naive rule (sort by prior traffic only) precision@50: 0.62
A real model needs to beat this trivial ordering, not just beat 0.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (one pseudonymized `content_id`), restricted to items that had measurable prior traffic. Below is the actual slice I'd build features and the target from.

In [7]:
cols_to_show = ["content_id", "client_id", "content_type", "clicks_prev_30d",
                "clicks_last_30d", "pct_change_30d", "recent_drop",
                "impressions_90d", "avg_position"]
cols_to_show = [c for c in cols_to_show if c in sub.columns]
sub[cols_to_show].head(10)

,content_id,client_id,content_type,clicks_prev_30d,clicks_last_30d,pct_change_30d,recent_drop,impressions_90d,avg_position
0,content_304f48230142,client_f369cb89fc,keyword article,13,2,-0.846154,1,3803,10.6
1,content_a1fb4e703a9e,client_4e07408562,keyword article,1,2,1.000000,0,15320,20.3
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3,1,-0.666667,1,12581,36.5
3,content_331d6c4de07b,client_19581e27de,keyword article,17,22,0.294118,0,11751,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2,10,4.000000,0,19140,44.0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,1,0,-1.000000,1,3970,8.5
8,content_5e6c160719bc,client_6208ef0f77,keyword article,8,9,0.125000,0,32574,46.0
10,content_d8ee6cc6d642,client_19581e27de,keyword article,117,112,-0.042735,0,20919,2.2
12,content_42fb2cad9ecf,client_6208ef0f77,keyword article,29,92,2.172414,0,7228,5.6
16,content_78bd1d4a1d4d,client_6208ef0f77,keyword article,5,9,0.800000,0,13848,8.9


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if statement rule treats every content item the same, but the actual rate of recent decline varies a lot by content type shown below. `comparison article` items show `recent_drop` 90% of the time, `feedly article` items 78.6%, but `keyword article` items only 51.6%, and they make up almost all of the data (11,476 of 11,759 items). A flat threshold either over flags the rare content types or under flags the dominant one; it can't weigh multiple signals together the way a model can. That's a tangled, multi-signal pattern — worth learning, not worth hand writing.

In [8]:
sub.groupby("content_type")["recent_drop"].agg(count="count", recent_drop_rate="mean").round(3)

,count,recent_drop_rate
content_type,,
comparison article,40,0.900
feedly article,243,0.786
keyword article,11476,0.516


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.